# mltu CRNN Training on Google Colab (GPU)

Trains the same CRNN+CTC model as `train_mltu.py` but on Colab's free GPU.
Expect ~30 min for 50 epochs vs ~17 hours on CPU.

**Prerequisites:**
1. Upload your `IAM_Words/` folder (containing `words.txt` + `words/` subdir) to Google Drive.
2. Set the paths in the **Config** cell below to match your Drive layout.
3. After training, copy `model.onnx` + `configs.yaml` from Drive to your local `models/mltu/`.

## 1. Install dependencies

In [11]:
!pip install -q mltu==1.2.5 pyyaml tf2onnx onnx

import tensorflow as tf
print(f"TensorFlow {tf.__version__}")
print(f"GPUs: {tf.config.list_physical_devices('GPU')}")

KeyboardInterrupt: 

## 2. Mount Google Drive

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Config

Edit these paths to match where you uploaded IAM_Words on Drive.

In [10]:
from pathlib import Path

# --- EDIT THESE ---
DATA_DIR = Path("/content/drive/MyDrive/IAM_Words")
OUT_DIR  = Path("/content/drive/MyDrive/models/mltu")
# ------------------

WORDS_TXT = DATA_DIR / "words.txt"
WORDS_DIR = DATA_DIR / "words"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert WORDS_TXT.exists(), f"{WORDS_TXT} not found. Upload IAM_Words to Drive first."
print(f"Dataset: {DATA_DIR}")
print(f"Output:  {OUT_DIR}")

Dataset: /content/drive/MyDrive/IAM_Words
Output:  /content/drive/MyDrive/models/mltu


## 4. Parse dataset

In [5]:
def load_samples():
    samples, vocab, max_len = [], set(), 0
    with open(WORDS_TXT, "r", encoding="utf-8") as fh:
        for line in fh:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.strip().split(" ")
            if len(parts) < 9 or parts[1] != "ok":
                continue
            word_id, label = parts[0], " ".join(parts[8:])
            a, b, *_ = word_id.split("-")
            img_path = WORDS_DIR / a / f"{a}-{b}" / f"{word_id}.png"
            if not img_path.exists() or img_path.stat().st_size == 0:
                continue
            samples.append([str(img_path), label])
            vocab.update(label)
            max_len = max(max_len, len(label))
    return samples, sorted(vocab), max_len

samples, vocab_list, max_word_length = load_samples()
vocab = "".join(vocab_list)
print(f"Samples: {len(samples)}, vocab: {len(vocab_list)}, max_word_length: {max_word_length}")

Samples: 0, vocab: 0, max_word_length: 0


## 5. Write configs.yaml

In [6]:
import yaml

HEIGHT = 32
WIDTH  = 256

with open(OUT_DIR / "configs.yaml", "w", encoding="utf-8") as fh:
    yaml.safe_dump({"vocab": vocab, "height": HEIGHT, "width": WIDTH, "max_text_length": max_word_length}, fh)

print(f"Saved {OUT_DIR / 'configs.yaml'}")

Saved /content/drive/MyDrive/models/mltu/configs.yaml


## 6. DataProvider + Augmentation

In [7]:
from mltu.annotations.images import CVImage
from mltu.augmentors import (
    RandomBrightness,
    RandomErodeDilate,
    RandomElasticTransform,
    RandomGaussianBlur,
    RandomRotate,
)
from mltu.dataProvider import DataProvider
from mltu.preprocessors import ImageReader
from mltu.transformers import ImageResizer, LabelIndexer, LabelPadding

BATCH_SIZE = 64

data_provider = DataProvider(
    dataset=samples,
    skip_validation=True,
    batch_size=BATCH_SIZE,
    data_preprocessors=[ImageReader(CVImage)],
    transformers=[
        ImageResizer(WIDTH, HEIGHT, keep_aspect_ratio=False),
        LabelIndexer(vocab),
        LabelPadding(max_word_length=max_word_length, padding_value=len(vocab)),
    ],
)

train_provider, val_provider = data_provider.split(split=0.9)

# Augmentors — training only (not applied to validation)
train_provider.augmentors = [
    RandomBrightness(random_chance=0.3, delta=100),
    RandomRotate(random_chance=0.3, angle=5),
    RandomErodeDilate(random_chance=0.3, kernel_size=(1, 1)),
    RandomGaussianBlur(random_chance=0.2, sigma=1.0),
    RandomElasticTransform(random_chance=0.2, alpha_range=(0, 0.05), sigma_range=(0.01, 0.02)),
]

print(f"Train batches: {len(train_provider)}, Val batches: {len(val_provider)}")

INFO:DataProvider:Skipping Dataset validation...


ValueError: Dataset must be iterable

## 7. Model definition (CRNN + CNN dropout)

In [ ]:
from tensorflow.keras.layers import (
    Activation, Add, BatchNormalization, Bidirectional,
    Conv2D, Dense, Dropout, Input, LSTM, Reshape,
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam


def residual_block(x, filters, strides=1):
    shortcut = x
    x = Conv2D(filters, (3, 3), strides=strides, padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(filters, (3, 3), padding='same', use_bias=False)(x)
    x = BatchNormalization()(x)
    if strides != 1 or shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, (1, 1), strides=strides, padding='same', use_bias=False)(shortcut)
        shortcut = BatchNormalization()(shortcut)
    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x


def build_model(input_dim, output_dim):
    inputs = Input(shape=input_dim, name='input')
    x = tf.keras.layers.Lambda(lambda img: img / 255.0)(inputs)

    x = residual_block(x, 16)
    x = residual_block(x, 16)
    x = Dropout(0.1)(x)
    x = residual_block(x, 32, strides=2)
    x = residual_block(x, 32)
    x = Dropout(0.15)(x)
    x = residual_block(x, 64, strides=2)
    x = residual_block(x, 64)
    x = Dropout(0.2)(x)

    x = tf.transpose(x, perm=[0, 2, 1, 3])
    shape = x.shape
    x = Reshape((shape[1], shape[2] * shape[3]))(x)

    x = Bidirectional(LSTM(128, return_sequences=True))(x)
    x = Dropout(0.25)(x)

    x = Dense(output_dim + 1, activation='softmax', name='output')(x)
    return Model(inputs=inputs, outputs=x)


model = build_model(
    input_dim=(HEIGHT, WIDTH, 3),
    output_dim=len(vocab),
)
model.summary(line_length=110)

## 8. Compile & train

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, TensorBoard
from tensorflow.keras.utils import Sequence
from mltu.tensorflow.callbacks import Model2onnx, TrainLogger
from mltu.tensorflow.losses import CTCloss
from mltu.tensorflow.metrics import CERMetric, WERMetric

EPOCHS = 50
LR = 1e-3


# Keras Sequence wrapper (needed if mltu's DataProvider isn't recognized).
class KerasSequenceProvider(Sequence):
    def __init__(self, provider):
        self.provider = provider
    def __len__(self):
        return len(self.provider)
    def __getitem__(self, index):
        return self.provider[index]
    def on_epoch_end(self):
        if hasattr(self.provider, 'on_epoch_end'):
            self.provider.on_epoch_end()


model.compile(
    optimizer=Adam(learning_rate=LR),
    loss=CTCloss(),
    metrics=[
        CERMetric(vocabulary=vocab),
        WERMetric(vocabulary=vocab),
    ],
)

callbacks = [
    EarlyStopping(monitor='val_CER', patience=10, verbose=1, mode='min'),
    ModelCheckpoint(
        filepath=str(OUT_DIR / 'model.h5'),
        monitor='val_CER', mode='min', save_best_only=True, verbose=1,
    ),
    ReduceLROnPlateau(monitor='val_CER', factor=0.9, patience=5, verbose=1, mode='min'),
    TensorBoard(log_dir=str(OUT_DIR / 'logs'), update_freq='epoch'),
    TrainLogger(str(OUT_DIR)),
    Model2onnx(saved_model_path=str(OUT_DIR / 'model.h5'), metadata={'vocab': vocab}),
]

# Try DataProvider directly; fall back to wrapper if Keras rejects it.
try:
    model.fit(
        train_provider, validation_data=val_provider,
        epochs=EPOCHS, callbacks=callbacks, workers=4,
    )
except ValueError:
    print("Falling back to KerasSequenceProvider wrapper...")
    model.fit(
        KerasSequenceProvider(train_provider),
        validation_data=KerasSequenceProvider(val_provider),
        epochs=EPOCHS, callbacks=callbacks, workers=4,
    )

print(f"\nDone. ONNX model: {OUT_DIR / 'model.onnx'}")
print(f"Configs: {OUT_DIR / 'configs.yaml'}")

## 9. Copy to local machine

After training completes, copy these two files from your Google Drive to your local project:

```
Drive: models/mltu/model.onnx    ->  local: models/mltu/model.onnx
Drive: models/mltu/configs.yaml  ->  local: models/mltu/configs.yaml
```

Then restart Streamlit and select **"mltu CRNN (IAM words)"** in the sidebar:

```bash
streamlit run app/streamlit_app.py
```

In [ ]:
# Verify output files exist
for f in ['model.onnx', 'configs.yaml', 'model.h5']:
    path = OUT_DIR / f
    exists = path.exists()
    size = f"{path.stat().st_size / 1e6:.1f} MB" if exists else "MISSING"
    print(f"  {f:20s}  {size}")